In [1]:
!hostname

ip-10-0-2-107


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import keras_tuner as kt
from keras_tuner import Objective
from imblearn.over_sampling import SMOTE
from collections import Counter
from imblearn.pipeline import Pipeline  
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

2025-10-30 06:54:14.741259: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-30 06:54:16.579539: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-30 06:54:20.195688: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Baseline model 

In [13]:
# Load data 
X_full_train = pd.read_csv("f5_train.csv")
X_test = pd.read_csv("f5_test.csv")

# Drop non-feature columns
for df in [X_full_train, X_test]:
    df.drop(columns=['transcript_id', 'transcript_position'], errors='ignore', inplace=True)

# Split labels
y_full_train = X_full_train.pop('label')
y_test = X_test.pop('label')

# Split 90% train into 80:20 (train : validation)
X_train, X_val, y_train, y_val = train_test_split(
    X_full_train, y_full_train,
    test_size=0.2,       
    stratify=y_full_train,
    random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (92596, 87), Val: (23150, 87), Test: (6092, 87)


In [14]:
# Scale features 

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [24]:
# Baseline model 
model_baseline = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])


model_baseline.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        AUC(curve='PR', name='pr_auc'),
        AUC(curve='ROC', name='roc_auc')
    ]
)

# Early stopping 
early_stop = EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

/home/ubuntu/.local/lib/python3.10/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:
# Train the model 
history_baseline = model_baseline.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.1440 - pr_auc: 0.2603 - roc_auc: 0.8567 - val_loss: 0.1212 - val_pr_auc: 0.4226 - val_roc_auc: 0.9071
Epoch 2/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1190 - pr_auc: 0.4431 - roc_auc: 0.9060 - val_loss: 0.1157 - val_pr_auc: 0.4578 - val_roc_auc: 0.9157
Epoch 3/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1151 - pr_auc: 0.4682 - roc_auc: 0.9132 - val_loss: 0.1155 - val_pr_auc: 0.4563 - val_roc_auc: 0.9169
Epoch 4/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1130 - pr_auc: 0.4867 - roc_auc: 0.9169 - val_loss: 0.1136 - val_pr_auc: 0.4777 - val_roc_auc: 0.9187
Epoch 5/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1111 - pr_auc: 0.4987 - roc_auc: 0.9200 - val_loss: 0.1129 - val_pr_auc: 0.4872 - val_roc_auc: 0.9214
Epoch 6/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1096 - pr_auc: 0.5115 - roc_auc: 0.9231 - val_loss: 0.1133 - val_pr_auc: 0.4878 - val_roc_auc: 0.9236
Epoch 7/50
724/7

In [26]:
# Evaluate on validation set 

y_val_probs = model_baseline.predict(X_val_scaled)
y_val_pred = (y_val_probs >= 0.5).astype(int)

print("\n=== Validation Set Evaluation ===")
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_val, y_val_probs))
print("PR-AUC:", average_precision_score(y_val, y_val_probs))

724/724 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

=== Validation Set Evaluation ===
Confusion Matrix:
 [[21908   175]
 [  719   348]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9682    0.9921    0.9800     22083
           1     0.6654    0.3261    0.4377      1067

    accuracy                         0.9614     23150
   macro avg     0.8168    0.6591    0.7089     23150
weighted avg     0.9543    0.9614    0.9550     23150

ROC-AUC: 0.9245471449389563
PR-AUC: 0.5002372387113483


In [27]:
# Evaluate on test set 

y_test_probs = model_baseline.predict(X_test_scaled)
y_test_pred = (y_test_probs >= 0.5).astype(int)

print("\n=== Test Set Evaluation ===")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, y_test_probs))
print("PR-AUC:", average_precision_score(y_test, y_test_probs))

191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  

=== Test Set Evaluation ===
Confusion Matrix:
 [[5886   66]
 [  97   43]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9838    0.9889    0.9863      5952
           1     0.3945    0.3071    0.3454       140

    accuracy                         0.9732      6092
   macro avg     0.6891    0.6480    0.6659      6092
weighted avg     0.9702    0.9732    0.9716      6092

ROC-AUC: 0.9018793202764976
PR-AUC: 0.30263064584480376
